In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score
import io, warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Mirror of app.py. Same CSV, same 8 features, same parameters, so the printed
# numbers match the README and the deployed Streamlit app.
#
# spotify_mau = Spotify MAU (millions), annual figure, same for all albums in a year
# 2015=80  2016=100  2017=120  2018=207  2019=248  2020=299
# 2021=365 2022=489  2023=574  2024=640  2025=715
# 2026 Jan-Jun = 761,  2026 Jul-Dec = 788  (Q3 2026 earnings)
#
# hype_score was removed from the data. It was scored after those albums released, so it
# carried hindsight the model could exploit. See "Model Revision, October 2026" in the README.
# lead_time = days from the first official announcement of the release date to the release.
# Every value was re-checked against announcement dates in October 2026.

csv_content = """artist,album,year,track_count,spotify_mau,lead_time,feature_track_count,years_since_last_album,prev_album_streams,first_week_streams,is_household_name
Ed Sheeran,Divide,2017,16,120,50,0,4,0,375,1
Ed Sheeran,Equals,2021,14,365,71,0,4,375,161,1
Drake,Scorpion,2018,25,207,16,3,1,0,559,1
Drake,CLB,2021,21,365,7,12,3,559,496,1
Drake,FATD,2023,23,574,14,10,2,496,405,1
Travis Scott,ASTROWORLD,2018,17,207,4,11,3,0,257,0
Travis Scott,Utopia,2023,19,574,6,13,5,257,457,0
Eminem,Revival,2017,19,120,17,8,4,0,117,1
Eminem,Kamikaze,2018,13,207,0,4,1,117,238,1
Eminem,TheDeathOfSlimShady,2024,19,640,11,8,6,238,254,1
The Weeknd,Starboy,2016,18,100,65,5,1,0,223,1
The Weeknd,AfterHours,2020,14,299,30,0,2,223,265,1
The Weeknd,HUT,2025,22,715,65,7,4,265,302,1
Justin Bieber,Purpose,2015,13,80,42,4,3,0,205,1
Justin Bieber,Justice,2021,16,365,21,8,2,205,252,1
SZA,SOS,2022,23,489,6,4,5,125,336,0
Taylor Swift,Midnights,2022,20,489,54,1,3,0,776,1
Taylor Swift,1989TV,2023,21,574,79,0,1,776,582,1
Taylor Swift,TTPD,2024,31,640,75,2,1,582,1173,1
Sabrina Carpenter,ShortNSweet,2024,12,640,81,0,3,0,352,0
Kendrick Lamar,DAMN,2017,14,120,3,3,2,0,227,1
Kendrick Lamar,MrMorale,2022,19,489,25,7,5,227,343,1
Kendrick Lamar,GNX,2024,12,640,0,4,2,343,363,1
LilWayne,CarterV,2018,23,207,2,10,7,0,285,0
Ariana Grande,ThanksUNext,2019,12,248,17,0,1,0,361,1
Ariana Grande,EternalSunshine,2024,13,640,51,0,5,361,342,1
Olivia Rodrigo,SOUR,2021,11,365,50,0,0,0,385,1
Olivia Rodrigo,GUTS,2023,12,574,74,0,2,385,284,1
Billie Eilish,HMHAS,2024,10,640,39,0,5,0,386,1
Beyonce,Renaissance,2022,16,489,43,2,6,0,173,1
Beyonce,CowboyCarter,2024,27,640,47,11,2,173,316,1
Dua Lipa,FutureNostalgia,2020,13,299,58,2,4,0,294,0
Dua Lipa,RadicalOptimism,2024,11,640,51,0,4,294,94,0
Adele,30,2021,12,365,37,0,6,0,310,1
Bad Bunny,UnVeranoSinTi,2022,23,489,4,8,1,0,503,1
Bad Bunny,NadieSabe,2023,22,574,4,8,1,503,469,1
Harry Styles,FineLine,2019,12,248,39,0,0,0,155,1
Harry Styles,HarrysHouse,2022,13,489,58,0,2,155,440,1
Post Malone,BBandB,2018,18,207,23,5,1,0,411,1
Post Malone,HollywoodsBleeding,2019,17,248,11,7,1,411,379,1
Karol G,MananaSeraBonito,2023,17,574,14,10,2,0,203,0
21 Savage,AmericanDream,2024,15,640,3,10,3,0,187,0
Tyler The Creator,CHROMAKOPIA,2024,14,640,11,7,3,0,340,0
Kanye West,DONDA,2021,27,365,23,23,6,0,442,1
Kanye West,VULTURES1,2024,15,640,24,0,3,442,251,1
Tate McRae,ThinkLater,2023,14,574,32,0,2,0,115,0
Metro Boomin,WDTY,2024,17,640,14,5,2,0,257,0
Morgan Wallen,Dangerous,2021,30,365,58,2,3,0,161,0
Juice WRLD,LegendsNeverDie,2020,22,299,4,5,1,0,303,0
Taylor Swift,LifeOfAShowgirl,2025,12,715,52,1,1,1173,887,1
Sabrina Carpenter,MansBestFriend,2025,13,715,79,0,1,352,297,0
Morgan Wallen,ImTheProblem,2025,37,715,57,5,2,161,253,0
Tate McRae,SoCloseToWhat,2025,16,715,99,2,2,115,188,0
Playboi Carti,MUSIC,2025,30,715,2,12,5,160,490,0
Lady Gaga,Mayhem,2025,14,715,39,2,5,173,219,1
Justin Bieber,Swag,2025,21,715,1,7,4,252,276.8,1
Harry Styles,KissAllTheTime,2026,14,761,50,0,4,440,205,1"""

# --- Reference-only history (NOT in training) ---------------------------------
# These earlier albums have real first-week numbers, but adding them to training
# measurably hurt holdout accuracy and skewed predictions upward (tested individually
# and combined; the dataset is too small and high-variance to absorb them safely).
# Kept here only as documentation and for manual "years since last album" lookups.
#
# Bad Bunny,UltimoTourDelMundo,2020,3,16,299,3,0,0,0,274,1
# Post Malone,TwelveCaratToothache,2022,1,14,489,37,6,3,379,142,1
# Tyler The Creator,CallMeIfYouGetLost,2021,1,16,365,10,12,2,0,133,0
# Billie Eilish,HappierThanEver,2021,0,16,365,74,0,2,0,178,1
# Taylor Swift,Folklore,2020,0,16,299,1,1,1,0,329,1
# Taylor Swift,Evermore,2020,0,15,299,1,3,0,329,254,1
# 21 Savage,SavageMode2,2020,1,15,299,7,5,2,0,108,0   (collab w/ Metro, kept separate)
# Metro Boomin,HeroesVillains,2022,1,15,489,14,10,0,0,226,0
# Ariana Grande,Sweetener,2018,0,15,207,21,2,2,0,174,1
# Harry Styles,HarryStyles,2017,0,10,120,30,0,0,0,66,0

# 8 features. hype_score dropped (hindsight leak); the three correlated history
# features collapsed into history_per_mau.
FEATS = ['track_count', 'spotify_mau', 'lead_time', 'feature_track_count',
         'years_since_last_album', 'history_per_mau',
         'is_household_name', 'household_x_tracks']

MAU_2026 = 788   # Jul-Dec 2026. Use 761 for a Jan-Jun release.

# --- prep ---
df = pd.read_csv(io.StringIO(csv_content)).sort_values(['artist','year']).reset_index(drop=True)
df['household_x_tracks'] = df['is_household_name'] * df['track_count']
df['spt'] = df['first_week_streams'] / df['track_count']
gm, gm_spt = df[df['year']<=2025]['first_week_streams'].median(), df['spt'].median()
df['prev_album_streams'] = df['prev_album_streams'].replace(0, gm)

def prior_mean(df, col, fallback):
    """Mean over the artist's STRICTLY EARLIER albums. Prevents leakage."""
    out = []
    for _, r in df.iterrows():
        prior = df[(df['artist']==r['artist']) & (df['year']<r['year'])]
        out.append(prior[col].mean() if len(prior) > 0 else fallback)
    return out

df['t_avg'] = prior_mean(df, 'first_week_streams', gm)
df['t_spt'] = prior_mean(df, 'spt', gm_spt)
for col, src in [('trailing_avg_per_mau','t_avg'), ('trailing_spt_per_mau','t_spt'),
                 ('prev_streams_per_mau','prev_album_streams')]:
    df[col] = df[src] / df['spotify_mau']

# Single history signal: last album and career average, averaged.
df['history_per_mau'] = (df['prev_streams_per_mau'] + df['trailing_avg_per_mau']) / 2
df['log_spm'] = np.log(df['first_week_streams'] / df['spotify_mau'])

train, hold, val26 = (df[df['year']<=2024].copy(),
                      df[df['year']==2025].copy(),
                      df[df['year']==2026].copy())

# --- LOOCV ---
MP = dict(n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=42)
X, y = train[FEATS], train['log_spm']
lp, la = [], []
for ti, te in LeaveOneOut().split(X):
    m = RandomForestRegressor(**MP); m.fit(X.iloc[ti], y.iloc[ti])
    lp.append(m.predict(X.iloc[te])[0]); la.append(y.iloc[te[0]])

r2     = r2_score(la, lp)
rp     = np.exp(np.array(lp)) * train['spotify_mau'].values
ra     = train['first_week_streams'].values
mape   = np.mean(np.abs(rp - ra) / ra * 100)
medape = np.median(np.abs(rp - ra) / ra * 100)
mae    = np.mean(np.abs(rp - ra))
r2_raw = r2_score(ra, rp)   # same predictions, scored on the raw stream scale

# Naive benchmark: always guess the median first-week total of the other albums.
base_pred = np.array([np.median(np.delete(ra, i)) for i in range(len(ra))])
base_ape  = np.abs(base_pred - ra) / ra * 100

model = RandomForestRegressor(**MP)
model.fit(X, y)

# --- helpers ---
def predict(params, mau):
    Xi = pd.DataFrame([params], columns=FEATS)
    tp = np.array([np.exp(t.predict(Xi)[0]) * mau for t in model.estimators_])
    return np.mean(tp), np.percentile(tp, 10), np.percentile(tp, 90)

def artist_params(artist, tracks, lead, feats, years, mau, before_year=2026):
    """Build a feature row for a new album. before_year restricts the artist's history
    to strictly earlier albums, matching how the training features were built."""
    full = df[df['artist']==artist].sort_values('year')
    a = full[full['year'] < before_year]
    is_hh = int(full.iloc[-1]['is_household_name']) if len(full) else 0
    prev = (a.iloc[-1]['first_week_streams'] if len(a) else gm) / mau
    tavg = (a['first_week_streams'].mean()   if len(a) else gm) / mau
    return [tracks, mau, lead, feats, years, (prev + tavg) / 2, is_hh, is_hh * tracks]

# --- 2026 case study: predicted ahead of each release -------------------------
# LOGGED values from the original model, kept as a record of what was actually
# predicted before each album came out. They are NOT recomputed here. The model has
# since been revised and would give different figures today; recomputing them would
# turn a forecast into a retrodiction.
CASE_STUDY = [
    ("DRAKE - ICEMAN",              479, 450.4, 22),
    ("OLIVIA RODRIGO - YSPSFAGSIL", 413, 394.7, 13),
    ("ARIANA GRANDE - PETAL",       408, 236.1, 12),
]

BOLD, BLUE, GREEN, RED, AMBER, RESET = "\033[1m", "\033[34m", "\033[32m", "\033[31m", "\033[33m", "\033[0m"
COL_W = [32, 9, 9, 21, 10, 9]

def fmt_row(label, m, tc, mape, actual):
    lo, hi = int(m*(1-mape/100)), int(m*(1+mape/100))
    d = (m - actual) / actual * 100
    c = GREEN if abs(d) < 15 else (AMBER if abs(d) < 30 else RED)
    return (label.ljust(COL_W[0])
            + (f"{BOLD}{BLUE}{int(m)}M{RESET}").ljust(COL_W[1] + len(BOLD+BLUE+RESET))
            + f"{m/tc:.1f}M".ljust(COL_W[2])
            + f"{lo}M-{hi}M".ljust(COL_W[3])
            + (f"{BOLD}{GREEN}{actual:.1f}M{RESET}").ljust(COL_W[4] + len(BOLD+GREEN+RESET))
            + f"{c}{BOLD}{d:+.1f}%{RESET}")

W = 100
hdr = ["ARTIST & ALBUM", "MODEL", "SPT", "RANGE (+/-" + f"{mape:.0f}" + "% MAPE)", "ACTUAL", "% DIFF"]

print("\n" + "="*W)
print(f"{'SUMMER 2026 CASE STUDY  (predicted ahead of each release)':^{W}}")
print("="*W)
print("  " + "".join(h.ljust(w) for h, w in zip(hdr, COL_W)))
print("  " + "-"*sum(COL_W))
for label, pred, actual, tc in CASE_STUDY:
    print("  " + fmt_row(label, pred, tc, mape, actual))
print("\n  Logged with the original 11-feature model. Not recomputed.")
print("  Two of three landed within 7%.")

print("\n" + "="*W + f"\n2025 HOLDOUT  ({len(hold)} albums, never seen in training)\n" + "="*W)
print(f"  {'ALBUM':<38}  {'PRED':>5}  {'ACTUAL':>6}  {'ERR':>9}\n  " + "-"*68)
herrs = []
for _, r in hold.iterrows():
    p,_,_ = predict(r[FEATS].tolist(), r['spotify_mau'])
    e = abs(p - r['first_week_streams']) / r['first_week_streams'] * 100
    herrs.append(e)
    over = 'over' if p > r['first_week_streams'] else 'under'
    print(f"  {r['artist']+' '+r['album']:<38}  {int(p):>4}M  {int(r['first_week_streams']):>5}M  {e:>7.1f}% ({over})")
hbase = np.mean(np.abs(np.median(ra) - hold['first_week_streams'].values) / hold['first_week_streams'].values * 100)
print(f"  >> Holdout MAPE: {np.mean(herrs):.1f}%   (always guessing the median album: {hbase:.1f}%)")

print("\n" + "="*W + "\n2026 VALIDATION  (known result, not trained on)\n" + "="*W)
for _, r in val26.iterrows():
    p,_,_ = predict(r[FEATS].tolist(), r['spotify_mau'])
    e = abs(p - r['first_week_streams']) / r['first_week_streams'] * 100
    over = 'over' if p > r['first_week_streams'] else 'under'
    print(f"  {r['artist']+' '+r['album']:<38}  {int(p):>4}M  {int(r['first_week_streams']):>5}M  {e:>7.1f}% ({over})")
print(f"  Note: {len(val26)} album. A single observation cannot estimate an error rate.")
print("  The 2025 holdout above is the real out-of-sample test.")

print("\n" + "="*W + "\nFEATURE IMPORTANCE\n" + "="*W)
for f, v in pd.Series(model.feature_importances_, index=FEATS).sort_values(ascending=False).items():
    print(f"  {f:<25}  {v:.3f}  " + "#"*int(v*44))

print("="*W)
print(f"\n  train={len(train)} rows | features={len(FEATS)} | R2(log)={r2:.3f} | "
      f"MAPE={mape:.1f}% | median={medape:.1f}% | MAE={mae:.0f}M")
print(f"  R2 on the raw stream scale = {r2_raw:.3f}. The {r2:.2f} above is measured on")
print("  log(streams/MAU). On the raw scale, a score at or below zero means no better than")
print("  always guessing the average album's stream count.")
print(f"  Benchmark, always guess the median album: MAPE={base_ape.mean():.1f}% | median={np.median(base_ape):.1f}%")
print(f"  Model:                                    MAPE={mape:.1f}% | median={medape:.1f}%")
print("="*W + "\n")
